# 第11章

In [ ]:
!apt-get install graphviz libgraphviz-dev pkg-config
!pip install pygraphviz==1.12
!pip install dowhy==0.11
!pip install econml==0.15

## リスト11.1

In [ ]:
import pygraphviz as pgv
from IPython.display import Image

causal_graph = """
digraph {
    "Prior Experience" -> "Player Skill Level";
    "Prior Experience" -> "Time Spent Playing";
    "Time Spent Playing" -> "Player Skill Level";
    "Guild Membership" -> "Side-quest Engagement";
    "Guild Membership" -> "In-game Purchases";
    "Player Skill Level" -> "Side-quest Engagement";
    "Player Skill Level" -> "In-game Purchases";
    "Time Spent Playing" -> "Side-quest Engagement";
    "Time Spent Playing" -> "In-game Purchases";
    "Side-quest Group Assignment" -> "Side-quest Engagement";
    "Customization Level" -> "Side-quest Engagement";
    "Side-quest Engagement" -> "Won Items";
    "Won Items" -> "In-game Purchases";
    "Won Items" -> "Total Inventory";
    "In-game Purchases" -> "Total Inventory";
}
"""
G = pgv.AGraph(string=causal_graph)
G.draw('/tmp/causal_graph.png', prog='dot')
Image('/tmp/causal_graph.png')

## リスト11.2

In [ ]:
import pandas as pd
data = pd.read_csv(
    "https://raw.githubusercontent.com/altdeep/causalAI/master/datasets/online_game_example_do_why.csv"
)
print(data.columns)

## リスト11.3

In [ ]:
from dowhy import CausalModel

model = CausalModel(
    data=data,
    treatment='Side-quest Engagement',
    outcome='In-game Purchases',
    graph=causal_graph
)

## リスト11.4

In [ ]:
identified_estimand = model.identify_effect()
print(identified_estimand)

## リスト11.5・11.6

In [ ]:
causal_estimate_reg = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.linear_regression",
    confidence_intervals=True
)
print(causal_estimate_reg)

## リスト11.7

In [ ]:
causal_estimate_strat = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.propensity_score_stratification",
    target_units="ate",
    confidence_intervals=True
)

print(causal_estimate_strat)

## リスト11.8

In [ ]:
causal_estimate_match = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.propensity_score_matching",
    target_units="ate",
    confidence_intervals=True
)
print(causal_estimate_match)

## リスト11.9

In [ ]:
causal_estimate_ipw = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.propensity_score_weighting",
    target_units = "ate",
    method_params={"weighting_scheme":"ips_weight"},
    confidence_intervals=True
)
print(causal_estimate_ipw)

## リスト11.10

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LassoCV
from sklearn.ensemble import GradientBoostingRegressor

gb_estimate = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.econml.dml.DML",
    control_value = 0,
    treatment_value = 1,
    method_params={
        "init_params":{
            'model_y':GradientBoostingRegressor(),
            'model_t': GradientBoostingRegressor(),
            "model_final":LassoCV(fit_intercept=False),
            'featurizer':PolynomialFeatures(degree=1, include_bias=False)
        },
        "fit_params":{}
    }
)
print(gb_estimate)

## リスト11.11

In [ ]:
from sklearn.ensemble import RandomForestRegressor
metalearner_estimate = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.econml.metalearners.TLearner",
    method_params={
        "init_params": {'models': RandomForestRegressor()},
        "fit_params": {}
    }
)

print(metalearner_estimate)

## リスト11.12

In [ ]:
causal_estimate_fd = model.estimate_effect(
    identified_estimand,
    method_name="frontdoor.two_stage_regression",
    target_units = "ate",
    method_params={"weighting_scheme": "ips_weight"},
    confidence_intervals=True
)
print(causal_estimate_fd)

## リスト11.13

In [ ]:
causal_estimate_iv = model.estimate_effect(
    identified_estimand,
    method_name="iv.instrumental_variable",
    method_params = {
        "iv_instrument_name": "Side-quest Group Assignment"
    },
    confidence_intervals=True
)
print(causal_estimate_iv)

## リスト11.14

In [ ]:
%%capture
causal_estimate_regdist = model.estimate_effect(
    identified_estimand,
    method_name="iv.regression_discontinuity",
    method_params={
        'rd_variable_name':'Customization Level',
        'rd_threshold_value':0.5,
        'rd_bandwidth': 0.15
    },
    confidence_intervals=True,
)

In [ ]:
print(causal_estimate_regdist)

## リスト11.15

In [ ]:
identified_estimand.set_identifier_method("frontdoor")
res_subset = model.refute_estimate(
    identified_estimand,
    causal_estimate_fd,
    method_name="data_subset_refuter",
    subset_fraction=0.8,
    num_simulations=100
)
print(res_subset)

## リスト11.16

In [ ]:
identified_estimand.set_identifier_method("backdoor")

res_random = model.refute_estimate(
    identified_estimand,
    gb_estimate,
    method_name="random_common_cause",
    num_simulations=100,
    n_jobs=2
)
print(res_random)

## リスト11.17

In [ ]:
identified_estimand.set_identifier_method("backdoor")

res_placebo = model.refute_estimate(
    identified_estimand,
    causal_estimate_ipw,
    method_name="placebo_treatment_refuter",
    placebo_type="permute",
    num_simulations=100
)

print(res_placebo)

## リスト11.18

In [ ]:
import numpy as np

coefficients = np.array([100.0, 50.0])
bias = 50.0
def linear_gen(df):
    subset = df[['guild_membership','player_skill_level']]
    y_new = np.dot(subset.values, coefficients) + bias
    return y_new

identified_estimand.set_identifier_method("frontdoor")
ref = model.refute_estimate(
    identified_estimand,
    causal_estimate_fd,
    method_name="dummy_outcome_refuter",
    outcome_function=linear_gen
)

res_dummy_outcome = ref[0]
print(res_dummy_outcome)

> **訳者補足**: dowhy 0.11 ではこの反証は機能しません。`outcome_function` は dowhy の公式ドキュメントには登場するものの実装上、無視されています。

## リスト11.19

In [ ]:
identified_estimand.set_identifier_method("backdoor")
res_unobserved = model.refute_estimate(
    identified_estimand,
    causal_estimate_strat,
    method_name="add_unobserved_common_cause"
)

print(res_unobserved)